# Modelo de interesse — o que ele procura, e os testes

Este notebook mostra, com dados reais do `studio/cache/`, **quais elementos o
scorer lê em cada frame** para achar as cenas de kill, e roda os testes do
pipeline (`studio/tests/pipeline/`).

Ordem:

1. Os **sinais base** por frame (`pipeline/features.py`)
2. As features derivadas que o scorer vê (`pipeline/derive.py`)
3. A **curva de interesse** de um clipe real
4. O que o **YOLO** conta como pessoa
5. **Frames extremos** de cada sinal (flash, vermelho, movimento, escuro)
6. **Combate × caminhada** — os `*_spike` e o `walk_flag`
7. **Aliado (verde) × inimigo** e a ROI de kill (`hit_center`)
8. **Testes** (`pytest`)

> Rode com o diretório de trabalho em `studio/` (`jupyter lab` a partir de lá),
> ou ajuste `STUDIO` na primeira célula. O cache pode ser de uma versão de
> features anterior — as colunas novas (`enemy_*`, `hit_center`,
> `friendly_green`) aparecem como `NaN` até re-extrair; a seção 7 recalcula ao
> vivo num frame.

In [ ]:
import sys, json, subprocess
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# acha studio/ a partir do cwd (raiz, studio/ ou studio/notebooks/)
STUDIO = next((p for p in [Path.cwd(), *Path.cwd().parents, Path.cwd() / "studio"]
               if (p / "config.py").exists() and p.name == "studio"), None)
assert STUDIO, f"não achei studio/config.py a partir de {Path.cwd()}"
sys.path.insert(0, str(STUDIO))

import config as C
from pipeline import features, derive, media

FR = STUDIO / "notebooks" / "_frames"      # jpgs temporários deste notebook
FR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110
print("ROOT :", C.ROOT)
print("fps  :", C.FPS_ANALYSIS, " largura análise:", C.ANALYSIS_WIDTH)
print("clipes_kill/ ->", C.KILL_CLIPS_DIR, "existe?", C.KILL_CLIPS_DIR.is_dir())

In [ ]:
# --- escolhe um clipe do cache: de preferência um que tenha vídeo no disco + score.json
srcs = json.loads((C.CACHE / "sources.json").read_text(encoding="utf8"))
pick = None
for key, rec in srcs.items():
    sha = rec["sha"]
    fname = key.split(":", 1)[0]
    on_disk = (C.ROOT / fname).exists()
    has_score = (C.CACHE / sha / "score.json").exists()
    has_feat = (C.CACHE / sha / "features.npz").exists()
    if has_feat and (pick is None or (on_disk, has_score) > (pick[3], pick[4])):
        pick = (fname, sha, rec, on_disk, has_score)

fname, SHA, rec, ON_DISK, HAS_SCORE = pick
VIDEO = C.ROOT / fname
feat = features.load_cached(SHA)
X, T, NAMES = feat["X"], feat["times"], feat["names"]
print(f"clipe    : {fname}")
print(f"sha      : {SHA[:12]}   vídeo no disco: {ON_DISK}   score.json: {HAS_SCORE}")
print(f"frames   : {len(T)}  ({T[-1]:.0f}s @ {C.FPS_ANALYSIS} fps)")

def frame(t, tag):
    "extrai 1 frame do vídeo no tempo t (s) e devolve a imagem RGB (ou None)"
    if not ON_DISK:
        return None
    dest = FR / f"{SHA[:8]}_{tag}.jpg"
    media.frame_jpeg(VIDEO, float(t), dest, width=480)
    return plt.imread(str(dest))

## 1 · Os sinais base por frame

`pipeline/features.py` decodifica a fonte **uma vez** por ffmpeg a 3 fps, escala
para 640 px de largura e calcula por frame (`NaN` = coluna nova que o cache
deste clipe ainda não tem):

In [ ]:
DESC = {
 "motion":          "|frame - frame anterior| médio na tela toda (0..1) — quanto a imagem mexeu",
 "center_motion":   "o mesmo, só no retângulo central (onde fica a mira/alvo)",
 "brightness":      "cinza médio / 255",
 "contrast":        "desvio-padrão do cinza / 128",
 "sat_mean":        "saturação média (HSV) / 255 — cena colorida × lavada",
 "red_frac":        "fração de pixels bem vermelhos (sangue, dano, hit)",
 "flash_frac":      "fração de pixels quase brancos (>230) — tiro, explosão, clarão",
 "dark_frac":       "fração de pixels quase pretos (<25) — tela escura, transição",
 "edge_density":    "fração de pixels de borda (Canny) — cena cheia de detalhe × limpa",
 "audio_rms":       "energia do áudio no frame (volume geral)",
 "audio_hi":        "energia acima de 1800 Hz — estalo de tiro, vidro, agudos",
 "audio_lo":        "energia abaixo de 250 Hz — explosão, passo, graves",
 "person_count":    "YOLO: nº de pessoas detectadas no frame",
 "person_conf":     "YOLO: confiança da melhor detecção",
 "person_area":     "YOLO: área da maior caixa / área do frame — inimigo perto",
 "person_center":   "YOLO: quão perto do centro está a pessoa mais central (1 = no centro)",
 "person_area_sum": "YOLO: soma das áreas de todas as caixas — muita gente / combate cheio",
 "hit_center":      "excesso de bordas num quadradinho no centro — os 'ticks' do hitmarker no acerto",
 "friendly_green":  "fração do verde de aliado na tela (contorno de teammate no HUD)",
 "enemy_count":     "caixas de pessoa SEM o contorno verde — os inimigos, alvos do kill",
 "enemy_area":      "área da maior caixa de inimigo / frame",
 "enemy_center":    "quão perto do centro está o inimigo mais central (1 = na mira)",
}
assert list(DESC) == list(features.FEATURE_NAMES) == NAMES
mean = np.nanmean(X, 0)
print(f"{'sinal':<16} {'média':>9}   descrição")
print("-" * 100)
for i, n in enumerate(NAMES):
    mv = f"{mean[i]:>9.4f}" if np.isfinite(mean[i]) else f"{'NaN':>9}"
    print(f"{n:<16} {mv}   {DESC[n]}")

## 2 · As features derivadas que o scorer realmente vê

"Interessante" é temporal, então `pipeline/derive.py` expande os sinais base:

- **`_wmax` / `_wmean`** — máximo e média numa janela de ~±1,5 s
- **`_spike`** — valor agora *menos* a média dos ~±4 s em volta, com piso 0.
  Sobe num kill (flash, vermelho, agudos, `hit_center`, área de
  inimigo); **não** sobe na caminhada.
- **`idle_flag`** — movimento central e áudio baixos e sustentados (menu / parado)
- **`walk_flag`** — movimento global acima da mediana do clipe **sem** nenhum
  `spike` de combate por perto → deslocamento, nada acontecendo

In [ ]:
Xa, AN = derive.augment(T, X, NAMES, C.FPS_ANALYSIS)
groups = {f"base ({len(NAMES)})": AN[:len(NAMES)],
          "_wmax":  [n for n in AN if n.endswith("_wmax")],
          "_wmean": [n for n in AN if n.endswith("_wmean")],
          "_spike": [n for n in AN if n.endswith("_spike")],
          "flags":  [n for n in AN if n.endswith("_flag")]}
for k, v in groups.items():
    print(f"{k:<12} {len(v):>2}  {', '.join(v)}")
print(f"\ntotal: {Xa.shape[1]} features  (matriz {Xa.shape})")
expected = (len(NAMES) + 2*len([c for c in derive.WINDOW_COLS if c in NAMES])
            + len([c for c in derive.SPIKE_COLS if c in NAMES]) + 2)
assert Xa.shape[1] == expected

## 3 · A curva de interesse de um clipe real

O scorer (`HistGradientBoostingClassifier`) dá uma probabilidade por frame; a
curva suavizada e a linha de corte saem em `cache/<sha>/score.json`. Os picos
acima do corte são os candidatos a virar corte de Short.

In [ ]:
sc_path = C.CACHE / SHA / "score.json"
if sc_path.exists():
    sc = json.loads(sc_path.read_text(encoding="utf8"))
    st, sm, thr = np.array(sc["times"]), np.array(sc["smooth"]), sc["threshold"]
    raw = np.array(sc["raw"])
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(st, raw, lw=.6, alpha=.4, label="raw")
    ax.plot(st, sm, lw=1.6, label="suavizada")
    ax.axhline(thr, color="crimson", ls="--", lw=1, label=f"corte {thr:.2f}")
    hot = sm >= thr
    ax.fill_between(st, 0, 1, where=hot, color="orange", alpha=.15, transform=ax.get_xaxis_transform())
    ax.set(xlabel="s", ylabel="interesse", title=f"{fname} — {hot.sum()} frames acima do corte")
    ax.legend(loc="upper right", ncol=4, fontsize=8); plt.show()
else:
    print("sem score.json para este clipe — rode:  python studio/pipeline/score.py \"%s\"" % fname)

## 4 · O que o YOLO conta como "inimigo na tela"

O detector é o `yolov8n` pré-treinado (COCO), usado **como está**, só na classe
`person`. Ele não é treinado com caixas suas — entra como *feature*: cinco
números por frame (`person_count/conf/area/center/area_sum`).

Pegar o frame de **maior `person_area_sum`** cru cai num menu (o YOLO "vê" uma
pessoa gigante com 0,31 de confiança na tela de matchmaking). Por isso os bons
frames abaixo são filtrados: **o modelo achou interessante** (`score ≥ corte`)
**e** a caixa tem tamanho de inimigo (`0,03 < person_area < 0,55`,
`person_conf ≥ 0,45`). O último painel mostra de propósito o erro do YOLO puro.

In [ ]:
from ultralytics import YOLO

sc_path = C.CACHE / SHA / "score.json"
sm = np.zeros(len(T)); thr = np.inf
if sc_path.exists():
    sc = json.loads(sc_path.read_text(encoding="utf8"))
    sm = np.interp(T, sc["times"], sc["smooth"]); thr = sc["threshold"]

col = lambda n: X[:, NAMES.index(n)]
pc, pa, pcf, dk = col("person_count"), col("person_area"), col("person_conf"), col("dark_frac")
plausible = (pc >= 1) & (pcf >= 0.45) & (pa > 0.03) & (pa < 0.55) & (dk < 0.5)
good = plausible & (sm >= thr)
if good.sum() < 3:                       # corte severo demais -> usa só o tamanho da caixa
    good = plausible
cand = np.where(good)[0]
picks = cand[np.argsort(X[cand, NAMES.index("person_area_sum")])[::-1]][:3]
# erro clássico: caixa enorme, confiança baixa, modelo NÃO achou interessante
fp = np.where((pa > 0.7) & (pcf < 0.5) & (sm < thr))[0]
bad = int(fp[np.argmax(pa[fp])]) if len(fp) else int(np.argmax(pa))

m = YOLO(str(C.YOLO_WEIGHTS))

def draw(ax, idx, ok):
    im = frame(float(T[idx]), ("ok" if ok else "fp") + str(idx))
    ax.axis("off")
    if im is None:
        ax.set_title("(sem vídeo no disco)"); return
    ax.imshow(im)
    r = m.predict(im, imgsz=C.YOLO_IMGSZ, classes=[0], verbose=False)[0]
    color = "lime" if ok else "red"
    if r.boxes is not None:
        for (x1, y1, x2, y2), cf in zip(r.boxes.xyxy.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            ax.add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=2))
            ax.text(x1, y1 - 3, f"{cf:.2f}", color=color, fontsize=9, weight="bold")
    tag = "" if ok else "YOLO ERRA — "
    ax.set_title(f"{tag}t={T[idx]:.1f}s  cnt={pc[idx]:.0f} conf={pcf[idx]:.2f} "
                 f"area={pa[idx]:.2f} score={sm[idx]:.2f}", fontsize=8)

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, idx in zip(axes[:3], picks):
    draw(ax, int(idx), ok=True)
draw(axes[3], bad, ok=False)
plt.tight_layout(); plt.show()

print(f"candidatos plausíveis: {len(cand)}   top-3 @ {np.round(T[picks],1)} s")
print("\n5 números do melhor frame:")
for n in ("person_count","person_conf","person_area","person_center","person_area_sum"):
    print(f"  {n:<16} {X[int(picks[0]), NAMES.index(n)]:.3f}")

## 5 · Frames extremos de cada sinal

O frame onde cada sinal é **máximo** no clipe — dá para ver na imagem o que o
número quis dizer.

In [ ]:
show = ["flash_frac", "red_frac", "center_motion", "dark_frac"]
fig, axes = plt.subplots(2, 2, figsize=(10, 6))
for ax, name in zip(axes.ravel(), show):
    j = int(np.argmax(X[:, NAMES.index(name)]))
    tj = float(T[j]); val = X[j, NAMES.index(name)]
    im = frame(tj, f"max_{name}")
    if im is not None:
        ax.imshow(im)
    ax.set_title(f"{name} máx = {val:.3f}  @ {tj:.1f}s", fontsize=10)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 6 · Combate × caminhada — `*_spike` e `walk_flag`

A diferença entre "tiroteio" e "andando" não está no movimento (os dois têm
movimento), e sim nos **picos locais** de flash / vermelho / agudos / área de
inimigo. `walk_flag` marca justamente movimento alto **sem** esses picos.

In [ ]:
combat = Xa[:, AN.index("flash_frac_spike")] + Xa[:, AN.index("audio_hi_spike")] \
       + Xa[:, AN.index("person_area_sum_spike")]
walk = Xa[:, AN.index("walk_flag")]
mot  = X[:, NAMES.index("motion")]

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(T, mot / max(mot.max(), 1e-6), lw=.8, label="motion (norm.)")
ax.plot(T, combat / max(combat.max(), 1e-6), lw=1.2, label="picos de combate (norm.)")
ax.fill_between(T, 0, walk, color="slategray", alpha=.25, label="walk_flag")
ax.set(xlabel="s", title="movimento existe nos dois; o pico de combate não")
ax.legend(loc="upper right", ncol=3, fontsize=8); plt.show()

i_fight = int(np.argmax(combat))
cand = np.where((walk > 0) & (combat < np.quantile(combat, 0.3)))[0]
i_walk = int(cand[len(cand)//2]) if len(cand) else int(np.argmax(walk))
fig, axs = plt.subplots(1, 2, figsize=(10, 3))
for ax, idx, lab in [(axs[0], i_fight, "COMBATE"), (axs[1], i_walk, "CAMINHADA")]:
    im = frame(float(T[idx]), f"{lab.lower()}")
    if im is not None: ax.imshow(im)
    ax.set_title(f"{lab}  t={T[idx]:.1f}s  combate={combat[idx]:.3f}  walk_flag={walk[idx]:.0f}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

## 7 · Aliado (verde) × inimigo, e as ROIs de kill

O HUD contorna **aliados de verde**; o **inimigo não tem** — é nele que a cena
de kill foca. Para cada caixa do YOLO o `features.py` mede a fração de verde
numa borda em volta (`FRIENDLY_BOX_PAD`) e chama de aliado se passar de
`FRIENDLY_GREEN_BOX_MIN`. Aqui isso é recalculado **ao vivo** num frame (o cache
do clipe pode ser de antes dessas features). A ROI `hit_center` (centro) também aparece desenhada. Bodycam não tem
killfeed na tela, então não há ROI pra isso.

> `FRIENDLY_GREEN_HSV_*` e `KILL_ROI_HITMARKER` em `config.py` são um chute — ajuste com
> um print real em `studio/model/kill_refs/`.

In [ ]:
import cv2
lo = np.array(C.FRIENDLY_GREEN_HSV_LO, np.uint8); hi = np.array(C.FRIENDLY_GREEN_HSV_HI, np.uint8)

# frame com mais pessoas detectadas no cache (proxy de "tem gente na tela")
j = int(np.nanargmax(np.nan_to_num(X[:, NAMES.index("person_count")])))
img = frame(float(T[j]), "enemy")
if img is None:
    print("(vídeo não está no disco — seção pula)")
else:
    bgr = img[:, :, ::-1].copy()
    H, W = bgr.shape[:2]
    hsv = cv2.cvtColor(bgr, cv2.COLOR_BGR2HSV)
    gmask = cv2.inRange(hsv, lo, hi)
    from ultralytics import YOLO
    m = YOLO(str(C.YOLO_WEIGHTS))
    r = m.predict(img, imgsz=C.YOLO_IMGSZ, classes=[0], verbose=False)[0]

    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].imshow(img); ax[0].set_title(f"aliado × inimigo  @ {T[j]:.1f}s"); ax[0].axis("off")
    x0, y0, x1, y1 = C.KILL_ROI_HITMARKER
    ax[0].add_patch(plt.Rectangle((x0*W, y0*H), (x1-x0)*W, (y1-y0)*H,
                    fill=False, color="yellow", lw=1.5, ls="--"))
    ax[0].text(x0*W, y0*H-4, "hit_center", color="yellow", fontsize=8)
    n_en = 0
    if r.boxes is not None:
        for b in r.boxes.xyxy.cpu().numpy():
            x1b, y1b, x2b, y2b = b.astype(int)
            g = features._green_ring_frac(bgr, x1b, y1b, x2b, y2b)
            friend = g >= C.FRIENDLY_GREEN_BOX_MIN
            col = "lime" if friend else "red"
            n_en += 0 if friend else 1
            ax[0].add_patch(plt.Rectangle((x1b, y1b), x2b-x1b, y2b-y1b, fill=False, color=col, lw=2))
            ax[0].text(x1b, y1b-3, f"{'ALIADO' if friend else 'INIMIGO'} v={g:.2f}",
                       color=col, fontsize=8, weight="bold")
    ax[1].imshow(gmask, cmap="Greens"); ax[1].axis("off")
    ax[1].set_title(f"máscara do verde de aliado — {gmask.mean()/255*100:.2f}% da tela  |  inimigos: {n_en}")
    plt.tight_layout(); plt.show()
    print("Se um inimigo ficou marcado ALIADO (ou vice-versa), ajuste FRIENDLY_GREEN_* / a máscara à direita.")

### 7·b — 5 frames aleatórios com as etiquetas do modelo

Sorteia 5 frames do clipe e mostra em cada um o que o modelo "vê": caixas do
YOLO classificadas em **ALIADO** (contorno verde do HUD) × **INIMIGO**, a ROI
`hit_center`, a cena derivada (`PARADO/MENU` · `CAMINHADA` ·
`COMBATE`) e se o score do frame passou do corte (`KILL?` × `nada`).

> Troque a semente em `np.random.default_rng(...)` para outros 5 frames.
> `hit_center` vem do cache — pode estar `0` se for de versão antiga das
> features.

In [ ]:
from ultralytics import YOLO

rng = np.random.default_rng(7)          # <- semente: mude para sortear outros frames
N = 5

# derivadas por frame (cena)
Xa, AN = derive.augment(T, X, NAMES, C.FPS_ANALYSIS)
walk   = Xa[:, AN.index("walk_flag")]
idle   = Xa[:, AN.index("idle_flag")]
combat = (Xa[:, AN.index("flash_frac_spike")] + Xa[:, AN.index("audio_hi_spike")]
          + Xa[:, AN.index("person_area_sum_spike")])
hi_combat = np.quantile(combat, 0.90)
hitc = np.nan_to_num(X[:, NAMES.index("hit_center")])

# curva do modelo (corte) se houver score.json
sm = np.zeros(len(T)); thr = np.inf
sc_path = C.CACHE / SHA / "score.json"
if sc_path.exists():
    _sc = json.loads(sc_path.read_text(encoding="utf8"))
    sm = np.interp(T, _sc["times"], _sc["smooth"]); thr = _sc["threshold"]

# sorteia: de preferência frames com gente na tela
_pc = np.nan_to_num(X[:, NAMES.index("person_count")])
pool = np.where(_pc >= 1)[0]
if len(pool) < N:
    pool = np.arange(len(T))
idxs = np.sort(rng.choice(pool, size=min(N, len(pool)), replace=False))

m = YOLO(str(C.YOLO_WEIGHTS))
fig, axes = plt.subplots(1, len(idxs), figsize=(4.2 * len(idxs), 4.6))
axes = np.atleast_1d(axes)
for ax, j in zip(axes, idxs):
    ax.axis("off")
    im = frame(float(T[j]), f"rnd{j}")
    if im is None:
        ax.set_title("(sem vídeo no disco)", fontsize=8); continue
    ax.imshow(im)
    H, W = im.shape[:2]
    bgr = im[:, :, ::-1].copy()

    x0, y0, x1, y1 = C.KILL_ROI_HITMARKER
    ax.add_patch(plt.Rectangle((x0 * W, y0 * H), (x1 - x0) * W, (y1 - y0) * H,
                               fill=False, color="yellow", lw=1.2, ls="--"))
    ax.text(x0 * W, y0 * H - 3, "hit_center", color="yellow", fontsize=7)

    r = m.predict(im, imgsz=C.YOLO_IMGSZ, classes=[0], verbose=False)[0]
    n_en = n_fr = 0
    if r.boxes is not None:
        for b in r.boxes.xyxy.cpu().numpy():
            x1b, y1b, x2b, y2b = b.astype(int)
            g = features._green_ring_frac(bgr, x1b, y1b, x2b, y2b)
            friend = g >= C.FRIENDLY_GREEN_BOX_MIN
            n_fr += int(friend); n_en += int(not friend)
            col = "lime" if friend else "red"
            ax.add_patch(plt.Rectangle((x1b, y1b), x2b - x1b, y2b - y1b,
                                       fill=False, color=col, lw=2))
            ax.text(x1b, y1b - 4, ("ALIADO" if friend else "INIMIGO") + f" v={g:.2f}",
                    color=col, fontsize=7, weight="bold")

    if idle[j] > 0:                 cena = "PARADO/MENU"
    elif walk[j] > 0:               cena = "CAMINHADA"
    elif combat[j] >= hi_combat:    cena = "COMBATE"
    else:                           cena = "-"
    kill = "KILL?" if sm[j] >= thr else "nada"
    ax.set_title(f"t={T[j]:.1f}s   [{kill}]  score={sm[j]:.2f}\n"
                 f"{cena}   inimigos={n_en}  aliados={n_fr}\n"
                 f"hit_center={hitc[j]:.2f}", fontsize=8)

fig.suptitle("5 frames aleatorios - como o modelo etiqueta cada um", y=1.02, fontsize=11)
plt.tight_layout(); plt.show()


## 8 · Treinar o modelo (ao vivo)

Esta célula **treina de verdade**: `pipeline.train.train()` colhe as linhas
(`clipes_kill/` + gravações), re-extrai features quando o cache é de versão
antiga (pode demorar minutos, YOLO na CPU), valida com `GroupKFold`, ajusta o
threshold e grava `studio/model/scorer.joblib` + `scorer_meta.json`.

> `RUN_TRAINING = False` → só mostra o modelo que já está salvo, sem treinar.
> Se o painel do studio estiver treinando ao mesmo tempo, deixe `False` para não
> competir pela escrita do arquivo.

In [ ]:
RUN_TRAINING = True

import time
if RUN_TRAINING:
    from pipeline import train as trainmod
    t0 = time.time()
    meta = trainmod.train(progress=lambda p, m: print(f"  {p*100:5.1f}%  {m}"))
    print(f"\n[ok] treino em {time.time()-t0:.0f}s")
else:
    meta = json.loads((STUDIO / "model" / "scorer_meta.json").read_text(encoding="utf8"))

v = meta.get("validation", {})
print(f"\nfonte      : {meta.get('source','?')}")
print(f"linhas     : {meta['n_rows']}  ({meta['n_pos']} pos / {meta['n_neg']} neg)")
print(f"features   : {meta['n_features']}   audio: {meta['audio_used']}   "
      f"clipes de kill: {meta.get('n_kill_clips', 0)}")
print(f"validacao  : {v.get('scheme','?')}")
print(f"ROC-AUC    : {v.get('roc_auc','?')}    avg precision: {v.get('avg_precision','?')}")
print(f"P / R / F1 : {v.get('precision','?')} / {v.get('recall','?')} / {v.get('f1','?')}")
print(f"threshold  : {meta['suggested_threshold']}")
print("\nfontes de treino:")
for s in meta["sources"]:
    print(f"  {s['kind']:<10} {s['file'][:44]:<44} {s.get('pos',0):>6} pos / {s.get('neg',0):>6} neg")

In [ ]:
tf = meta.get("top_features", [])
if tf:
    fig, ax = plt.subplots(figsize=(9, max(2, 0.4 * len(tf))))
    ax.barh([t["feature"] for t in tf][::-1], [t["importance"] for t in tf][::-1], color="#4c78a8")
    ax.set_title("peso das features (permutation importance)")
    plt.tight_layout(); plt.show()
else:
    print("sem importancia de features no meta")

### A nova curva de interesse no clipe de exemplo

Reaplica o modelo recém-treinado no mesmo clipe da seção 3 e mostra a curva +
onde ela passa do corte.

In [ ]:
if RUN_TRAINING and VIDEO.exists():
    from pipeline import score as scoremod
    sc = scoremod.score_source(VIDEO, force=True,
                               progress=lambda p, m: print(f"  {p*100:5.1f}%  {m}") if int(p*100) % 20 == 0 else None)
    st, sm, thr = np.array(sc["times"]), np.array(sc["smooth"]), sc["threshold"]
    hot = sm >= thr
    fig, ax = plt.subplots(figsize=(12, 3.2))
    ax.plot(st, np.array(sc["raw"]), lw=.5, alpha=.35, label="raw")
    ax.plot(st, sm, lw=1.6, label="suavizada")
    ax.axhline(thr, color="crimson", ls="--", lw=1, label=f"corte {thr:.2f}")
    ax.fill_between(st, 0, 1, where=hot, color="orange", alpha=.15, transform=ax.get_xaxis_transform())
    ax.set(xlabel="s", ylabel="interesse",
           title=f"{VIDEO.name} — modelo novo — {hot.sum()} frames acima do corte")
    ax.legend(loc="upper right", ncol=4, fontsize=8); plt.show()
else:
    print("pulei (RUN_TRAINING=False ou vídeo fora do disco)")

## 9 · Testes do pipeline

`studio/tests/pipeline/` — regras de colheita do dataset (clipes de kill +
gravações; `--raw-only` = só gravações) e a expansão das features do
`derive.augment`.

In [ ]:
p = subprocess.run([sys.executable, "-m", "pytest", "tests/pipeline", "-v"],
                   cwd=str(STUDIO), capture_output=True, text=True)
print(p.stdout[-4000:])
print(p.stderr[-2000:])
assert p.returncode == 0, "testes falharam"

### Como retreinar com este modelo

1. Deixe na raiz ao menos uma **gravação com cortes aprovados**
   (`*/projeto/edicao.json`) — é ela que dá os negativos.
2. Corte da própria gravação **clipes crus de kill de 10–15 s** (sem editar, sem
   música) e jogue em `BODYCAM/clipes_kill/`.
3. (recomendado) Ponha 2–3 prints de um kill em `studio/model/kill_refs/` para
   calibrar `KILL_ROI_*` / `FRIENDLY_GREEN_*` em `config.py`.
4. Painel **0 · Modelo de interesse → Treinar / retreinar**, ou:
   ```
   python studio/pipeline/train.py
   ```
   `--raw-only` (ou o checkbox) ignora `clipes_kill/` e treina só nas gravações.